# Problema 1: Tienda con inventario

*Enunciado adaptado de los problemas 4 y 5 del Banco de Problemas de Aprendizaje por Refuerzo (MDT-Módulo 3, IIND-3518). Al final del cuaderno se explica qué se modificó.*


**Enunciado:** Una tienda mantiene un inventario $S_t \in \{0, 1, 2\}$. Al comenzar el día elige un pedido $A_t \in \{0, \dots, 2 - S_t\}$, que llega de inmediato. La demanda del día es $D_t \in \{0, 1\}$, con probabilidad $1/2$ para cada valor. Las ventas, el inventario siguiente y la recompensa son

$$Y_t = \min\{S_t + A_t,\ D_t\}, \qquad S_{t+1} = S_t + A_t - Y_t, \qquad R_{t+1} = 5Y_t - 2A_t - S_{t+1}.$$

Es decir, cada unidad vendida deja 5, cada unidad pedida cuesta 2 y cada unidad que queda en inventario al final del día cuesta 1. El negocio opera indefinidamente, empieza con $S_0 = 1$ y usa $\gamma = 0{,}9$.

- (a) Formule el problema para Q-learning: estados, acciones factibles y qué observa el algoritmo después de cada decisión.
- (b) Entrene Q-learning con $\alpha = 0{,}005$, $\varepsilon = 0{,}2$, $10^6$ pasos y semilla 2026. Presente la tabla $Q$ y la política greedy aprendida.
- (c) Calcule la política óptima de forma exacta con *value iteration* y compárela con la aprendida en (b).
- (d) Interprete la política óptima en cada estado.

---
## Preparación: importar la librería

In [1]:
import sys
import os.path
import numpy as np
# context for the examples
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..')))
from jmarkov.mdp.qlearning import qlearning
from jmarkov.mdp.dtmdp import dtmdp

---
## Solución (a)

**Aquí se usa la librería:** el problema se describe con las tres funciones que recibe la clase `qlearning`.

- **Estados:** el inventario al comenzar el día, $s \in \{0, 1, 2\}$. La función `initial_state` devuelve $S_0 = 1$.
- **Acciones factibles:** los pedidos que no superan la capacidad de 2 unidades, $a \in \{0, \dots, 2 - s\}$. Lo expresa la función `feasible_actions`.
- **Qué observa el algoritmo:** la función `step` simula un día. Sortea la demanda, calcula ventas, inventario siguiente y recompensa, y devuelve solo **(estado siguiente, recompensa)**. Q-learning aprende de esa observación; nunca recibe las probabilidades de la demanda.

In [3]:
GAMMA = 0.9


def initial_state(rng):
    return 1  # S_0 = 1


def feasible_actions(state):
    # Pedidos posibles: a = 0, ..., 2 - s.
    return tuple(range(0, 3 - state))


def step(state, action, rng):
    demanda = int(rng.integers(0, 2))           # D_t = 0 o 1, probabilidad 1/2
    ventas = min(state + action, demanda)        # Y_t
    siguiente = state + action - ventas          # S_{t+1}
    recompensa = 5 * ventas - 2 * action - siguiente   # R_{t+1}
    return siguiente, recompensa


model = qlearning(initial_state, feasible_actions, step, discount_factor=GAMMA)   

for s in (0, 1, 2):
    print(f"Estado s = {s}: acciones factibles {feasible_actions(s)}")

Estado s = 0: acciones factibles (0, 1, 2)
Estado s = 1: acciones factibles (0, 1)
Estado s = 2: acciones factibles (0,)


---
## Solución (b)

**Aquí se usa la librería:** `model.solve(...)` entrena Q-learning durante $10^6$ pasos, siguiendo una sola trayectoria desde $S_0 = 1$ y eligiendo las acciones con $\varepsilon$-greedy. Devuelve el valor estimado de cada estado ($\max_a Q(s,a)$) y la política greedy. La tabla $Q$ completa queda en `model.Q`.

In [4]:
valores_ql, politica_ql = model.solve(
    n_steps=1_000_000, alpha=0.005, epsilon=0.2, minimize=False, seed=2026)   

print("Tabla Q aprendida:")
for s in (0, 1, 2):
    fila = "   ".join(f"Q({s},{a}) = {model.Q[s][a]:7.3f}" for a in feasible_actions(s))
    print(f"  s = {s}:   {fila}")

print("\nPolítica greedy aprendida:", {s: politica_ql[s] for s in (0, 1, 2)})
print("Valores estimados:        ", {s: round(valores_ql[s], 3) for s in (0, 1, 2)})

Tabla Q aprendida:
  s = 0:   Q(0,0) =   8.205   Q(0,1) =   8.941   Q(0,2) =   6.912
  s = 1:   Q(1,0) =  10.986   Q(1,1) =   8.880
  s = 2:   Q(2,0) =  10.653

Política greedy aprendida: {0: 1, 1: 0, 2: 0}
Valores estimados:         {0: 8.941, 1: 10.986, 2: 10.653}


---
## Solución (c)

**Aquí se usa jMarkov, pero no la clase de Q-learning:** la solución exacta se obtiene con la clase `dtmdp`, que aplica *value iteration*. A diferencia de Q-learning, `dtmdp` sí necesita el modelo completo:

- **Matriz de transición por pedido:** la probabilidad de pasar de $s$ a $s'$. Con demanda 0 el inventario queda en $s + a$; con demanda 1 baja una unidad si hay inventario.
- **Recompensa esperada** de cada par $(s, a)$: el promedio de las recompensas con demanda 0 y 1.
- **Pedidos no factibles** ($s + a > 2$): como `dtmdp` usa las mismas acciones en todos los estados, se les asigna una recompensa muy negativa ($-10^6$), para que nunca se elijan.

In [5]:
estados = np.array([0, 1, 2])
acciones = np.array([0, 1, 2])

matrices = {a: np.zeros((3, 3)) for a in acciones}
recompensas = np.full((3, 3), -1e6)   # filas: estados; columnas: acciones

for s in estados:
    for a in acciones:
        if s + a > 2:                     # pedido no factible
            matrices[a][s, s] = 1.0
            continue
        recompensas[s, a] = 0.0
        for demanda in (0, 1):            # cada demanda con probabilidad 1/2
            ventas = min(s + a, demanda)
            siguiente = s + a - ventas
            matrices[a][s, siguiente] += 0.5
            recompensas[s, a] += 0.5 * (5 * ventas - 2 * a - siguiente)

mdp = dtmdp(estados, acciones, matrices, recompensas, GAMMA)                       
valores_exactos, politica_exacta = mdp.solve(1e-12, minimize=False, method="value_iteration")   

print(f"{'Estado':>6} {'Política exacta':>16} {'Política Q-learning':>20} {'V exacto':>10} {'V Q-learning':>13}")
for i, s in enumerate(estados):
    print(f"{s:>6} {int(politica_exacta[s]):>16} {politica_ql[s]:>20} "
          f"{valores_exactos[i]:>10.3f} {valores_ql[s]:>13.3f}")

Estado  Política exacta  Política Q-learning   V exacto  V Q-learning
     0                1                    1      9.000         8.941
     1                0                    0     11.000        10.986
     2                0                    0     10.818        10.653


Las dos políticas coinciden en los tres estados. Los valores de Q-learning son estimaciones: se acercan a los exactos, pero no son iguales, porque se obtienen de una trayectoria simulada finita y con tasa de aprendizaje constante.

In [6]:
# Ganancia de seguir la política aprendida, calculada de forma exacta:
# se resuelve (I - gamma * P_pi) V = r_pi.
P_pi = np.array([matrices[politica_ql[s]][s] for s in estados])        # fila de cada estado con su acción
r_pi = np.array([recompensas[s, politica_ql[s]] for s in estados])     # recompensa esperada de esa acción
ganancia_ql = np.linalg.solve(np.eye(len(estados)) - GAMMA * P_pi, r_pi)

print(f"{'Estado':>6} {'Ganancia máxima':>16} {'Ganancia política Q-learning':>29}")
for i, s in enumerate(estados):
    print(f"{s:>6} {valores_exactos[i]:>16.3f} {ganancia_ql[i]:>29.3f}")

Estado  Ganancia máxima  Ganancia política Q-learning
     0            9.000                         9.000
     1           11.000                        11.000
     2           10.818                        10.818


---
## Solución (d)

**Aquí no se usa la librería:** es la interpretación de la política obtenida en (b) y (c).

- **Inventario 0 → pedir 1 unidad.** Sin inventario no se puede vender. Pedir una unidad permite atender la demanda si llega; pedir dos no tiene sentido, porque la demanda es como máximo 1 y la segunda unidad solo generaría costo de pedido y de inventario.
- **Inventario 1 → no pedir.** La unidad disponible basta para atender la demanda máxima del día. Pedir otra solo agregaría costo.
- **Inventario 2 → no pedir.** Es la única acción factible, porque el inventario ya está en el máximo.

En resumen, la política óptima mantiene una unidad disponible al comenzar cada día: pide solo cuando el inventario está en cero.

---
## Nota sobre la adaptación

En el problema 5 original, la tienda se analiza en episodios de cuatro días y la política se aprende con control Monte Carlo. La librería resuelve problemas continuos con descuento en una sola trayectoria. Por eso aquí el negocio opera indefinidamente y la política se aprende con Q-learning.

El inventario, los pedidos factibles, la demanda, las ventas, la recompensa y el factor de descuento son los del enunciado original.